# 03. Cartesian Genetic Programming (CGP) & NSGA-II Pareto Optimization

Discover how Darwin-Evolab uses Cartesian Genetic Programming (CGP) to optimize digital arithmetic logic units (ALUs) across 4 competing physical objectives: Correctness, Dynamic Power, Delay, and Area.


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from evolab.pareto import NSGA2Engine, build_silicon_multiobjective_evaluator
from evolab.cgp_logic import create_random_cgp_genome
from evolab.genome import Individual
import random
print("CGP and NSGA-II modules loaded successfully.")


## 1. Setup 4-Objective Silicon Evaluation
We evaluate a 1-bit Half-Adder on 4 objectives simultaneously.


In [ ]:
truth_table = [
    ((0, 0), (0, 0)),
    ((0, 1), (1, 0)),
    ((1, 0), (1, 0)),
    ((1, 1), (0, 1)),
]
objectives, eval_vector_fn = build_silicon_multiobjective_evaluator(truth_table)
for obj in objectives:
    print(f"Objective: {obj.name:15s} | Direction: {obj.direction:8s} | Weight: {obj.weight}")


## 2. Run NSGA-II Non-Dominated Sorting


In [ ]:
rng = random.Random(42)
pop = [Individual(create_random_cgp_genome(2, 2, 8, rng=rng), species="spec_logic") for _ in range(12)]

engine = NSGA2Engine(
    objectives=objectives,
    evaluate_vector_fn=eval_vector_fn,
    population_size=12,
    generations=5,
    seed=42,
)
result = engine.run(initial_population=pop, generations=5)
print(f"Total Non-Dominated Pareto Solutions: {len(result['pareto_front'])}")
for i, sol in enumerate(result['pareto_front'][:3]):
    print(f"Solution {i+1}: Objective Vector = {sol['objectives']}")
